# SBE37 preview data plot

Use this notebook to enter an `inst_deploy_id`, inspect the source file, and preview the data plot without running the full pipeline.

## Usage

1. Set the working directory if needed.
2. Choose the instrument type.
3. Enter the `inst_deploy_id`.
4. Run the ID lookup cell.
5. Run the source inspection cell to see the file start and end timestamps.
6. Run the preview plot cell.

In [ ]:
import os
import sys
from pathlib import Path
import importlib

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

TOOLS_DIR = Path.cwd().resolve().parent
if not (TOOLS_DIR / 'tools').exists():
    candidate = Path.cwd().resolve()
    if (candidate / 'tools').exists():
        TOOLS_DIR = candidate
    elif (candidate / 'mooring_proc' / 'tools').exists():
        TOOLS_DIR = candidate / 'mooring_proc'

if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

import pandas as pd
import tools.database_lookup as database_lookup
from tools.parsers.read_sbe37 import read_sbe37
from tools.helpers import plot_data_by_qc


## Working directory

Edit this once if you want relative paths to resolve from a different folder.

In [ ]:
# Working directory
working_directory = "/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data"
os.chdir(working_directory)
print(f"Working directory: {os.getcwd()}")


## Setup

Set the deployment identifier and instrument type. The instrument type controls the default preview variables.

In [ ]:
# Required deployment identifier from the metadata table.
inst_deploy_id = "275"

# Choose the instrument type so the preview plot knows which variables to show.
instrument = "SBE37"

# Optional raw source override. Leave as None to use metadata resolution.
proc1_source_path = None

# Optional manual overrides for the preview plot.
preview_plot_variables = None
preview_plot_flags = None
preview_zoom_to_good = True

instrument_preview_defaults = {
    "SBE37": ["TEMP", "CNDC", "PSAL", "PRES"],
    "SBE26": ["TEMP", "PSAL", "PRES"],
    "RBRQ": ["TEMP", "PRES"],
    "AQD": ["UCUR", "VCUR", "PRES", "TEMP"],
    "SIG500": ["TEMP", "PRES"],
}


## ID lookup

This cell resolves the selected deployment and prints the metadata row.

In [ ]:
def fmt_utc(value):
    timestamp = pd.to_datetime(value, utc=True, errors="coerce")
    if pd.isna(timestamp):
        return ""
    return timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")

database_lookup = importlib.reload(database_lookup)
inst_deploy_id = str(inst_deploy_id).strip()
instrument = str(instrument).strip().upper()

selected_metadata_row = None
selected_metadata_cfg = None
selected_metadata_lines = []

_, selected_metadata_row, selected_metadata_cfg, selected_metadata_lines = database_lookup.get_instrument_context(
    None,
    inst_deploy_id,
    deployment_id=None,
)

print(f"Metadata for inst_deploy_id={inst_deploy_id}:")
for line in selected_metadata_lines:
    print(line)


## Inspect source file

This loads the raw source file and prints the file start/end timestamps before you do anything else.

In [ ]:
proc1_source_preview = read_sbe37(proc1_source_path, config={"metadata_row": selected_metadata_row.to_dict()})
proc1_source_path = proc1_source_preview["input_path"]
proc1_source_frame = proc1_source_preview["dataframe"]
proc1_source_start = proc1_source_frame.index.min()
proc1_source_end = proc1_source_frame.index.max()

print(f"Resolved source path: {proc1_source_path}")
print(f"Source start timestamp: {fmt_utc(proc1_source_start)}")
print(f"Source end timestamp: {fmt_utc(proc1_source_end)}")
print(f"Deployment window start: {fmt_utc(selected_metadata_row.get('time_coverage_start'))}")
print(f"Deployment window end: {fmt_utc(selected_metadata_row.get('time_coverage_end'))}")


## Preview plot

This plots the source data using a simple instrument-aware default variable list.

In [ ]:
preview_variables = preview_plot_variables or instrument_preview_defaults.get(instrument, [])
source_plot = proc1_source_preview["dataset"]
print(f"Instrument: {instrument}")
print(f"Preview variables: {preview_variables}")

preview_figure = plot_data_by_qc(
    source_plot,
    variables=preview_variables,
    flags_to_plot=preview_plot_flags,
    y_zoom_to_good=preview_zoom_to_good,
    title=f"{instrument} preview plot: {inst_deploy_id}",
)
preview_figure.show()
